# Demo — Limpieza de datos (Pandas) con un CSV de 5,000 registros (simple v2)

Archivo: **`dataset_5k_sucio_simple_v2.csv`**

Simplificaciones:
- **Genero:** solo valores `M`/`F` con variacion en mayusculas/minusculas y espacios.
- **Fechas:** solo `YYYY-MM-DD` y `DD/MM/YYYY`.
- **Precios:** solo coma vs punto decimal (y algunos vacios).


In [1]:
# Dependencias
# !pip install pandas numpy

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 1) Cargar CSV y evidencia 'antes'

In [2]:
from pathlib import Path

DATA_PATH = Path("dataset_5k_sucio_simple_v2.csv")
df = pd.read_csv(DATA_PATH)

df_before = df.copy(deep=True)
print("Shape:", df.shape)
display(df.head(10))


Shape: (5000, 7)


,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria
0,502609,Paola Morales,m,2026-02-27,NaN,Antigua,Retail
1,500588,ANA DIAZ,f,2026-02-24,371.80,Villa Nueva,Services
2,502408,Maria Paz,m,2026-01-15,"373,33",quetzaltenango,retail
3,501688,Ana Mejia,f,2026-01-18,361.23,Quetzaltenango,Education
4,502916,Carlos Mendez,f,29/01/2026,"339,01",Amatitlan,Retail
5,504200,Oscar Rojas,m,2026-01-09,138.22,Antigua,RETAIL
6,502029,Isabel Ramirez,m,2026-02-05,"166,31",Villa Nueva,SERVICES
7,501366,Pedro Herrera,m,12/02/2026,"405,62",Amatitlan,Education
8,502614,Patricia Soto,F,2026-01-11,222.88,guatemala,Retail
9,502794,Pablo Perez,M,2026-02-06,"287,37",Escuintla,Retail


## 2) Diagnostico: duplicados y nulos/vacios

In [3]:
print("Duplicados exactos:", int(df.duplicated().sum()))

print("\nNulos por columna (NaN reales):")
display(df.isna().sum())

print("\nVacios (strings vacios) por columna:")
empty_counts = {}
for col in df.columns:
    if df[col].dtype == "object":
        empty_counts[col] = int((df[col].astype("string").str.strip() == "").sum())
display(pd.Series(empty_counts).sort_values(ascending=False))


Duplicados exactos: 100

Nulos por columna (NaN reales):


id_cliente          0
nombre              0
genero             60
fecha_registro      0
gasto_q           505
ciudad            157
categoria           0
dtype: int64


Vacios (strings vacios) por columna:


Series([], dtype: object)

## 3) Limpieza (simple)

- Tokens vacios a `NaN`.
- Texto: `strip`, espacios multiples, `title`.
- **Genero:** `str.strip().str.upper()` para dejar solo `M`/`F`.
- Fechas: `pd.to_datetime(..., dayfirst=True)`.
- Precios: `replace(',', '.')` + `to_numeric`.


In [4]:
EMPTY_TOKENS = {"", " ", "  ", "NA", "N/A", "null", "None", "-", "--"}

df_clean = df.copy()

# 3.1) Tokens vacios a NaN
for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        s = df_clean[col].astype("string").str.strip()
        df_clean[col] = s.where(~s.isin(list(EMPTY_TOKENS)), other=pd.NA)

# 3.2) Texto
df_clean["nombre"] = (
    df_clean["nombre"].astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.title()
)

df_clean["ciudad"] = (
    df_clean["ciudad"].astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.title()
)

df_clean["categoria"] = df_clean["categoria"].astype("string").str.strip().str.title()

# 3.3) Genero (simple): dejar M/F en mayuscula
df_clean["genero_std"] = df_clean["genero"].astype("string").str.strip().str.upper()

# 3.4) Fechas (simple)
df_clean["fecha_registro_dt"] = pd.to_datetime(df_clean["fecha_registro"], errors="coerce", dayfirst=True)

# 3.5) Precios (solo coma vs punto)
precio = df_clean["gasto_q"].astype("string").str.replace(",", ".", regex=False)
df_clean["gasto_q_num"] = pd.to_numeric(precio, errors="coerce")

df_clean.head(10)


C:\Users\javie\AppData\Local\Temp\ipykernel_27736\2895939951.py:32: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_clean["fecha_registro_dt"] = pd.to_datetime(df_clean["fecha_registro"], errors="coerce", dayfirst=True)


,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria,genero_std,fecha_registro_dt,gasto_q_num
0,502609,Paola Morales,m,2026-02-27,NaN,Antigua,Retail,M,2026-02-27,<NA>
1,500588,Ana Diaz,f,2026-02-24,371.80,Villa Nueva,Services,F,2026-02-24,371.8
2,502408,Maria Paz,m,2026-01-15,"373,33",Quetzaltenango,Retail,M,2026-01-15,373.33
3,501688,Ana Mejia,f,2026-01-18,361.23,Quetzaltenango,Education,F,2026-01-18,361.23
4,502916,Carlos Mendez,f,29/01/2026,"339,01",Amatitlan,Retail,F,NaT,339.01
5,504200,Oscar Rojas,m,2026-01-09,138.22,Antigua,Retail,M,2026-01-09,138.22
6,502029,Isabel Ramirez,m,2026-02-05,"166,31",Villa Nueva,Services,M,2026-02-05,166.31
7,501366,Pedro Herrera,m,12/02/2026,"405,62",Amatitlan,Education,M,NaT,405.62
8,502614,Patricia Soto,F,2026-01-11,222.88,Guatemala,Retail,F,2026-01-11,222.88
9,502794,Pablo Perez,M,2026-02-06,"287,37",Escuintla,Retail,M,2026-02-06,287.37


## 4) Duplicados: evidenciar y eliminar

In [5]:
dup_before = int(df_clean.duplicated().sum())
df_nodup = df_clean.drop_duplicates()
dup_after = int(df_nodup.duplicated().sum())

print("Duplicados antes de eliminar:", dup_before)
print("Duplicados despues de eliminar:", dup_after)
print("Shape despues:", df_nodup.shape)


Duplicados antes de eliminar: 100
Duplicados despues de eliminar: 0
Shape despues: (4900, 10)


## 5) Tratamiento de nulos (criterio simple y documentado)

In [6]:
df_after = df_nodup.copy()

df_after["ciudad"] = df_after["ciudad"].fillna("Desconocido")
df_after["genero_std"] = df_after["genero_std"].fillna("No informado")

if df_after["gasto_q_num"].isna().any():
    df_after["gasto_q_num"] = df_after["gasto_q_num"].fillna(df_after["gasto_q_num"].median())

print("Nulos por columna (despues del tratamiento):")
display(df_after.isna().sum().sort_values(ascending=False).head(20))


Nulos por columna (despues del tratamiento):


fecha_registro_dt    1481
gasto_q               497
genero                 60
id_cliente              0
nombre                  0
fecha_registro          0
ciudad                  0
categoria               0
genero_std              0
gasto_q_num             0
dtype: int64

## 6) Evidencia: antes vs despues (muestras)

In [7]:
print("ANTES (df_before)")
display(df_before.head(10))

print("DESPUES (df_after)")
display(df_after.head(10))


ANTES (df_before)


,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria
0,502609,Paola Morales,m,2026-02-27,NaN,Antigua,Retail
1,500588,ANA DIAZ,f,2026-02-24,371.80,Villa Nueva,Services
2,502408,Maria Paz,m,2026-01-15,"373,33",quetzaltenango,retail
3,501688,Ana Mejia,f,2026-01-18,361.23,Quetzaltenango,Education
4,502916,Carlos Mendez,f,29/01/2026,"339,01",Amatitlan,Retail
5,504200,Oscar Rojas,m,2026-01-09,138.22,Antigua,RETAIL
6,502029,Isabel Ramirez,m,2026-02-05,"166,31",Villa Nueva,SERVICES
7,501366,Pedro Herrera,m,12/02/2026,"405,62",Amatitlan,Education
8,502614,Patricia Soto,F,2026-01-11,222.88,guatemala,Retail
9,502794,Pablo Perez,M,2026-02-06,"287,37",Escuintla,Retail


DESPUES (df_after)


,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria,genero_std,fecha_registro_dt,gasto_q_num
0,502609,Paola Morales,m,2026-02-27,NaN,Antigua,Retail,M,2026-02-27,251.8
1,500588,Ana Diaz,f,2026-02-24,371.80,Villa Nueva,Services,F,2026-02-24,371.8
2,502408,Maria Paz,m,2026-01-15,"373,33",Quetzaltenango,Retail,M,2026-01-15,373.33
3,501688,Ana Mejia,f,2026-01-18,361.23,Quetzaltenango,Education,F,2026-01-18,361.23
4,502916,Carlos Mendez,f,29/01/2026,"339,01",Amatitlan,Retail,F,NaT,339.01
5,504200,Oscar Rojas,m,2026-01-09,138.22,Antigua,Retail,M,2026-01-09,138.22
6,502029,Isabel Ramirez,m,2026-02-05,"166,31",Villa Nueva,Services,M,2026-02-05,166.31
7,501366,Pedro Herrera,m,12/02/2026,"405,62",Amatitlan,Education,M,NaT,405.62
8,502614,Patricia Soto,F,2026-01-11,222.88,Guatemala,Retail,F,2026-01-11,222.88
9,502794,Pablo Perez,M,2026-02-06,"287,37",Escuintla,Retail,M,2026-02-06,287.37


## 7) Exportar CSV limpio (opcional)

In [8]:
from pathlib import Path
OUT_PATH = Path("dataset_5k_limpio_simple_v2.csv")
df_after.to_csv(OUT_PATH, index=False, encoding="ascii")
print("Exportado:", OUT_PATH.resolve())


Exportado: C:\Users\javie\Desktop\Universidad\actividades_ss2\tareas\tarea1\dataset_5k_limpio_simple_v2.csv
